# Member 4 - Logistic Regression
## ML Assignment: FIFA Player Position Prediction

- **Algorithm:** Logistic Regression (Multi-class)
- **Dataset:** FIFA 22 Complete Player Dataset
- **Source:** Kaggle - FIFA 22 Players
- **Task:** Predict player position - ATTACKER, MIDFIELDER, or DEFENDER

This is a standalone version. No external files needed. Data is downloaded automatically.


## 1. Introduction - What is Logistic Regression?

Logistic Regression is a classification algorithm. Even though it has "regression" in the name, it is used to **predict categories**, not numbers.

It works by:
1. Taking the input features and calculating a score (called `z`)
2. Passing that score through a **softmax function** to get probabilities for each class
3. Picking the class with the **highest probability**

**Why use it for FIFA position prediction?**

- It is easy to understand - the model shows which stats matter most
- It gives a confidence score for each position
- It trains very fast, even on thousands of players
- It is a good starting point to compare with other algorithms
- One limitation: it can only draw a straight-line boundary between classes


## 2. Import Librarie


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

# Machine learning tools
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Plot settings
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')

print('All libraries imported successfully!')

## 3. Load and Prepare the Dataset

The dataset comes from Kaggle and contains stats for over 18,000 real football players from FIFA 22.

**Features used to train the model:**

- `age` - player age
- `height_cm` - height in centimetres
- `weight_kg` - weight in kilograms
- `overall` - skill rating (0–100)
- `potential` - potential rating (0–100)
- `pace` - speed
- `shooting` - shooting accuracy
- `passing` - passing accuracy
- `dribbling` - ball control
- `defending` - defensive skill
- `physic` - physical strength

**Target label:** ATTACKER, MIDFIELDER, or DEFENDER


In [ ]:
import urllib.request, io

# Dataset URL (FIFA 22 players from public GitHub mirror)
DATA_URL = 'https://raw.githubusercontent.com/datasets/football-datasets/main/fifa22_players.csv'

# Features we will use for prediction
FEATURES = ['age','height_cm','weight_kg','overall','potential',
            'pace','shooting','passing','dribbling','defending','physic']

def position_group(pos):
    # Map a FIFA position string to one of three groups
    if not isinstance(pos, str):
        return None
    pos = pos.upper().strip()
    attackers  = {'ST','CF','LW','RW','LF','RF','LS','RS','LAM','RAM','CAM'}
    midfielders= {'CM','CDM','LM','RM','LCM','RCM','LDM','RDM'}
    defenders  = {'CB','LB','RB','LWB','RWB','LCB','RCB'}
    if pos in attackers:   return 'ATTACKER'
    if pos in midfielders: return 'MIDFIELDER'
    if pos in defenders:   return 'DEFENDER'
    return None

def make_synthetic_fifa_data(n=3000, seed=42):
    # Create realistic synthetic player data when real CSV is unavailable
    rng = np.random.default_rng(seed)
    rows = []

    # Average stat profiles for each position group
    profiles = {
        'ATTACKER':   dict(pace=75, shooting=78, passing=65, dribbling=76,
                          defending=32, physic=65, overall=72),
        'MIDFIELDER': dict(pace=68, shooting=62, passing=76, dribbling=72,
                          defending=55, physic=66, overall=73),
        'DEFENDER':   dict(pace=65, shooting=38, passing=60, dribbling=58,
                          defending=78, physic=74, overall=72),
    }

    for label, prof in profiles.items():
        for _ in range(n // 3):
            age    = int(rng.normal(26, 4))
            height = int(rng.normal(181, 6))
            weight = int(rng.normal(76, 8))
            ovr    = int(rng.normal(prof['overall'], 8))
            row = dict(
                age        = np.clip(age, 17, 42),
                height_cm  = np.clip(height, 160, 205),
                weight_kg  = np.clip(weight, 55, 110),
                overall    = np.clip(ovr, 45, 99),
                potential  = np.clip(ovr + int(rng.normal(4, 3)), 45, 99),
                pace       = np.clip(int(rng.normal(prof['pace'],       12)), 20, 99),
                shooting   = np.clip(int(rng.normal(prof['shooting'],   12)), 20, 99),
                passing    = np.clip(int(rng.normal(prof['passing'],    10)), 20, 99),
                dribbling  = np.clip(int(rng.normal(prof['dribbling'],  10)), 20, 99),
                defending  = np.clip(int(rng.normal(prof['defending'],  12)), 20, 99),
                physic     = np.clip(int(rng.normal(prof['physic'],     10)), 20, 99),
                position_group = label
            )
            rows.append(row)

    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

# Try loading real dataset, fall back to synthetic if unavailable
df = None
try:
    raw = pd.read_csv(DATA_URL)
    raw['position_group'] = raw['player_positions'].apply(
        lambda x: position_group(str(x).split(',')[0]) if pd.notna(x) else None
    )
    df = raw.dropna(subset=FEATURES + ['position_group'])
    df = df[df['position_group'] != 'GK']
    df = df[FEATURES + ['position_group']]
    print(f'Real dataset loaded: {len(df):,} players')
except Exception:
    df = make_synthetic_fifa_data(n=3000)
    print(f'Synthetic FIFA-style dataset created: {len(df):,} players')

print(f'\nDataset shape : {df.shape}')
print(f'Features      : {FEATURES}')
print(f'Target classes: {df["position_group"].unique().tolist()}')
print()
print('Class distribution:')
print(df['position_group'].value_counts().to_string())
print()
df[FEATURES].describe().round(2)